# Step 2: Hybrid-Grained Multimodal-Fused Reranking (OMGM Baseline)

This notebook implements **OMGM's Stage 2 Multimodal Reranker** on the 1,000 InfoSeek test subset:
1. **Loads Stage 1 Outputs:** Reads `result.json` containing top-20 candidate Wikipedia entities per query.
2. **Maps Candidate Images & Sections:** Links candidate entities to their extracted representative images and Wikipedia article sections from the 100k KB.
3. **Loads Pretrained Q-Former Reranker:** Instantiates the official `qformer_IT2IT_reranker` and loads the released weights (`model_23912.pth` / `reranker_model.pth`).
4. **Computes Multimodal Fusion Scores:** Computes late-interaction token-to-token Max-Sum similarity between $(I_q, T_q)$ and $(I_e, sec_h^e)$.
5. **Score Fusion:** Combines Stage 1 and Stage 2 similarities: $\text{Score} = \alpha \cdot \text{Sim}_{\text{step1}} + (1 - \alpha) \cdot \text{Sim}_{\text{step2}}$ (default $\alpha = 0.9$).
6. **Outputs for Stage 3:** Saves reranked entities and fine-grained section similarities (`sec_sim`) required by the Stage 3 text reranker and generator.

### Cell 1: Environment Verification & GPU Diagnostics

In [2]:
import os
import sys
import gc
import glob
import json
import time
import zipfile
import torch
import PIL.Image
from tqdm import tqdm

print(f"PyTorch Version:  {torch.__version__}")
print(f"CUDA Available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:      {torch.cuda.get_device_name(0)}")
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total GPU VRAM:   {total_mem:.2f} GB")
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


PyTorch Version:  2.11.0+cu128
CUDA Available:   True
Device Name:      Tesla T4
Total GPU VRAM:   14.56 GB


### Cell 2: Path Configuration & Dataset Autodetection
Configures paths to Stage 1 output, Knowledge Base JSON, test CSV, candidate images archive, and Q-Former reranker checkpoint.

In [28]:
# ---------------------------------------------------------------------------
# Kaggle Dataset & Path Configuration for Stage 2 Reranking
# ---------------------------------------------------------------------------
import os
import glob

BASE_DS_DIR = "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds"
ALT_DS_DIR = "/kaggle/input/agentic-vqa-ds"

def resolve_path(exact_path, search_patterns=None, is_dir=False):
    """Helper to resolve paths with primary check and fallback glob search."""
    if os.path.exists(exact_path):
        if is_dir and os.path.isdir(exact_path):
            return exact_path
        elif not is_dir and os.path.isfile(exact_path):
            return exact_path
        elif not is_dir and os.path.exists(exact_path):
            return exact_path

    alt_path = exact_path.replace(BASE_DS_DIR, ALT_DS_DIR)
    if os.path.exists(alt_path):
        return alt_path

    if search_patterns:
        for pat in search_patterns:
            matches = glob.glob(pat, recursive=True)
            for m in matches:
                if is_dir and os.path.isdir(m):
                    return m
                elif not is_dir and os.path.isfile(m):
                    return m
                elif os.path.exists(m):
                    return m
    return exact_path

# 1. FAISS Index (Stage 1 Artifact)
FAISS_INDEX_PATH = resolve_path(
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/InfoSeek_summary_faiss_index/summary_llama3.index",
    search_patterns=[
        "/kaggle/input/**/summary_llama3.index",
        "/kaggle/input/**/InfoSeek_summary_faiss_index/*.index",
        "/kaggle/input/**/InfoSeek_summary_faiss_index*",
    ],
    is_dir=False
)

# 2. Wikipedia 100k Knowledge Base
KB_PATH = resolve_path(
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_100k_wiki/wiki_100_dict_v4.json",
    search_patterns=[
        "/kaggle/input/**/wiki_100_dict_v4.json",
        "/kaggle/input/**/infoseek_100k_wiki/*.json",
        "/kaggle/input/**/wiki_100k*.json",
    ],
    is_dir=False
)

# 3. Query Images Directory (1,000 InfoSeek test images)
QUERY_IMG_DIR = resolve_path(
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_val_1000_images/infoseek_val_1000_images",
    search_patterns=[
        "/kaggle/input/**/infoseek_val_1000_images/infoseek_val_1000_images",
        "/kaggle/input/**/infoseek_val_1000_images",
    ],
    is_dir=True
)

# 4. Pretrained Q-Former Checkpoint (model_23912.pth inside reranker_model.pth folder)
RERANKER_CKPT_PATH = resolve_path(
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/reranker_model.pth/reranker_model.pth/model_23912.pth",
    search_patterns=[
        "/kaggle/input/**/model_23912.pth",
        "/kaggle/input/**/reranker_model.pth/**/model_23912.pth",
        "/kaggle/input/**/*reranker*.pth",
    ],
    is_dir=False
)
if os.path.isdir(str(RERANKER_CKPT_PATH)):
    for root, dirs, files in os.walk(str(RERANKER_CKPT_PATH)):
        for f in files:
            if f.endswith(".pth"):
                RERANKER_CKPT_PATH = os.path.join(root, f)
                break

# 5. Wiki Image Metadata Output CSVs (13 parts inside wiki_img_output/output)
WIKI_IMG_OUTPUT_DIR = resolve_path(
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/wiki_img_output/output",
    search_patterns=[
        "/kaggle/input/**/wiki_img_output/output",
        "/kaggle/input/**/wiki_img_output*",
    ],
    is_dir=True
)

# 6. Candidate Wiki Images Directory (13 split folders inside wiki_imgs/wiki_imgs/full/wiki_image_split)
WIKI_IMGS_DIR = resolve_path(
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/wiki_imgs/wiki_imgs/full/wiki_image_split",
    search_patterns=[
        "/kaggle/input/**/wiki_imgs/wiki_imgs/full/wiki_image_split",
        "/kaggle/input/**/wiki_image_split",
        "/kaggle/input/**/wiki_imgs/wiki_imgs",
        "/kaggle/input/**/wiki_imgs",
    ],
    is_dir=True
)

# 7. Candidate Wiki Images Filtered CSV (candidate_wiki_images_max3.csv)
CANDIDATE_IMAGES_CSV = resolve_path(
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/candidate_wiki_images_max3.csv",
    search_patterns=[
        "/kaggle/input/**/candidate_wiki_images_max3.csv",
        "/kaggle/input/**/candidate_wiki_images*.csv",
    ],
    is_dir=False
)

# 8. 1000 Sampled Test Questions CSV (infoseek_test_sample_1000.csv)
SAMPLE_FILE = resolve_path(
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_test_sample_1000.csv",
    search_patterns=[
        "/kaggle/input/**/infoseek_test_sample_1000.csv",
        "/kaggle/input/**/infoseek_test*.csv",
    ],
    is_dir=False
)

# 9. Stage 1 Retrieval Results JSON (result.json)
STAGE1_RESULT_PATH = resolve_path(
    "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/result.json",
    search_patterns=[
        "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/result.json",
        "/kaggle/input/**/result.json",
        "/kaggle/working/step1_results/result.json",
    ],
    is_dir=False
)

# Output directory for Stage 2
OUTPUT_DIR = "/kaggle/working/step2_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("="*60)
print("STAGE 2 PIPELINE PATHS CONFIGURED")
print("="*60)
print(f"1. FAISS Index:         {FAISS_INDEX_PATH} (Exists: {os.path.exists(str(FAISS_INDEX_PATH))})")
print(f"2. Knowledge Base JSON: {KB_PATH} (Exists: {os.path.exists(str(KB_PATH))})")
print(f"3. Query Images Dir:    {QUERY_IMG_DIR} (Exists: {os.path.isdir(str(QUERY_IMG_DIR))})")
print(f"4. Reranker Weights:    {RERANKER_CKPT_PATH} (Exists: {os.path.exists(str(RERANKER_CKPT_PATH))})")
print(f"5. Wiki Image Meta Dir: {WIKI_IMG_OUTPUT_DIR} (Exists: {os.path.isdir(str(WIKI_IMG_OUTPUT_DIR))})")
print(f"6. Wiki Candidate Imgs: {WIKI_IMGS_DIR} (Exists: {os.path.isdir(str(WIKI_IMGS_DIR))})")
print(f"7. Candidate CSV (max3): {CANDIDATE_IMAGES_CSV} (Exists: {os.path.exists(str(CANDIDATE_IMAGES_CSV))})")
print(f"8. Sample Questions:    {SAMPLE_FILE} (Exists: {os.path.exists(str(SAMPLE_FILE))})")
print(f"9. Stage 1 Result JSON: {STAGE1_RESULT_PATH} (Exists: {os.path.exists(str(STAGE1_RESULT_PATH))})")
print(f"   Output Directory:    {OUTPUT_DIR}")
print("="*60)

STAGE 2 PIPELINE PATHS CONFIGURED
1. FAISS Index:         /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/InfoSeek_summary_faiss_index/summary_llama3.index (Exists: True)
2. Knowledge Base JSON: /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_100k_wiki/wiki_100_dict_v4.json (Exists: True)
3. Query Images Dir:    /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_val_1000_images/infoseek_val_1000_images (Exists: True)
4. Reranker Weights:    /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/reranker_model.pth/reranker_model.pth/model_23912.pth (Exists: True)
5. Wiki Image Meta Dir: /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/wiki_img_output/output (Exists: True)
6. Wiki Candidate Imgs: /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/wiki_imgs/wiki_imgs/full/wiki_image_split (Exists: True)
7. Candidate CSV (max3): /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/candidate_wiki_images_max3.csv (Exists: True)
8. Sample Questions:    /kaggl

### Cell 3: Dependencies & OMGM Repository Setup
Installs required packages and ensures OMGM's customized LAVIS codebase is in `sys.path`.

In [29]:
# Install lightweight requirements if not already present in environment
!pip install -q timm decord fairscale webdataset iopath omegaconf

import os
import sys
import base64

# ---------------------------------------------------------------------------
# Locate or clone OMGM repository
# ---------------------------------------------------------------------------
OMGM_REPO_DIR = None
candidate_omgm_dirs = [
    "/kaggle/working/OMGM",
    "/kaggle/input/omgm-code/OMGM",
    "/kaggle/input/omgm/OMGM Main Baseline",
    "/kaggle/input/omgm-main-baseline/OMGM Main Baseline",
]
for p in candidate_omgm_dirs:
    if os.path.isdir(p) and os.path.exists(os.path.join(p, "lavis")):
        OMGM_REPO_DIR = p
        break

if OMGM_REPO_DIR is None:
    target_repo = "/kaggle/working/OMGM"
    if not os.path.isdir(target_repo):
        print(f"Cloning official OMGM repository to {target_repo}...")
        !git clone -q https://github.com/ChaoLinAViy/OMGM {target_repo}
    if os.path.isdir(target_repo):
        OMGM_REPO_DIR = target_repo

if OMGM_REPO_DIR and OMGM_REPO_DIR not in sys.path:
    sys.path.insert(0, OMGM_REPO_DIR)
print(f"OMGM Repository mounted from: {OMGM_REPO_DIR}")

# ---------------------------------------------------------------------------
# Patch LAVIS source files for modern transformers compatibility
# ---------------------------------------------------------------------------
PATCHER_B64 = "IiIiCnBhdGNoX2xhdmlzX2NvbXBhdC5weQotLS0tLS0tLS0tLS0tLS0tLS0tLQpQYXRjaGVzIE9NR00ncyBMQVZJUyBzb3VyY2UgZmlsZXMgaW4tcGxhY2UgdG8gYmUgY29tcGF0aWJsZSB3aXRoCm1vZGVybiBIdWdnaW5nIEZhY2UgdHJhbnNmb3JtZXJzICh2NC40MCsgLyB2NS54KS4KIiIiCgppbXBvcnQgb3MKaW1wb3J0IHJlCmltcG9ydCBzeXMKCkxBVklTX0RJUiA9IHN5cy5hcmd2WzFdIGlmIGxlbihzeXMuYXJndikgPiAxIGVsc2UgIk9NR00gTWFpbiBCYXNlbGluZS9sYXZpcyIKClNISU0gPSAnJycKIyAtLS0tIHRyYW5zZm9ybWVycyBjb21wYXRpYmlsaXR5IHNoaW0gKGF1dG8tcGF0Y2hlZCkgLS0tLQppbXBvcnQgdG9yY2gKaW1wb3J0IHRvcmNoLm5uIGFzIG5uCgpkZWYgX2xhdmlzX2ZpbmRfcHJ1bmVhYmxlX2hlYWRzX2FuZF9pbmRpY2VzKGhlYWRzLCBuX2hlYWRzLCBoZWFkX3NpemUsIGFscmVhZHlfcHJ1bmVkX2hlYWRzKToKICAgIG1hc2sgPSB0b3JjaC5vbmVzKG5faGVhZHMsIGhlYWRfc2l6ZSkKICAgIGhlYWRzID0gc2V0KGhlYWRzKSAtIGFscmVhZHlfcHJ1bmVkX2hlYWRzCiAgICBmb3IgaGVhZCBpbiBoZWFkczoKICAgICAgICBoZWFkID0gaGVhZCAtIHN1bSgxIGlmIGggPCBoZWFkIGVsc2UgMCBmb3IgaCBpbiBhbHJlYWR5X3BydW5lZF9oZWFkcykKICAgICAgICBtYXNrW2hlYWRdID0gMAogICAgbWFzayA9IG1hc2sudmlldygtMSkuY29udGlndW91cygpLmVxKDEpCiAgICBpbmRleCA9IHRvcmNoLmFyYW5nZShsZW4obWFzaykpW21hc2tdLmxvbmcoKQogICAgcmV0dXJuIGhlYWRzLCBpbmRleAoKZGVmIF9sYXZpc19wcnVuZV9saW5lYXJfbGF5ZXIobGF5ZXIsIGluZGV4LCBkaW09MCk6CiAgICBpbmRleCA9IGluZGV4LnRvKGxheWVyLndlaWdodC5kZXZpY2UpCiAgICBXID0gbGF5ZXIud2VpZ2h0LmluZGV4X3NlbGVjdChkaW0sIGluZGV4KS5jbG9uZSgpLmRldGFjaCgpCiAgICBpZiBsYXllci5iaWFzIGlzIG5vdCBOb25lOgogICAgICAgIGIgPSBsYXllci5iaWFzLmNsb25lKCkuZGV0YWNoKCkgaWYgZGltID09IDEgZWxzZSBsYXllci5iaWFzW2luZGV4XS5jbG9uZSgpLmRldGFjaCgpCiAgICBuZXdfc2l6ZSA9IGxpc3QobGF5ZXIud2VpZ2h0LnNpemUoKSkKICAgIG5ld19zaXplW2RpbV0gPSBsZW4oaW5kZXgpCiAgICBuZXdfbGF5ZXIgPSBubi5MaW5lYXIobmV3X3NpemVbMV0sIG5ld19zaXplWzBdLCBiaWFzPWxheWVyLmJpYXMgaXMgbm90IE5vbmUpLnRvKGxheWVyLndlaWdodC5kZXZpY2UpCiAgICBuZXdfbGF5ZXIud2VpZ2h0LnJlcXVpcmVzX2dyYWQgPSBGYWxzZQogICAgbmV3X2xheWVyLndlaWdodC5jb3B5XyhXLmNvbnRpZ3VvdXMoKSkKICAgIG5ld19sYXllci53ZWlnaHQucmVxdWlyZXNfZ3JhZCA9IFRydWUKICAgIGlmIGxheWVyLmJpYXMgaXMgbm90IE5vbmU6CiAgICAgICAgbmV3X2xheWVyLmJpYXMucmVxdWlyZXNfZ3JhZCA9IEZhbHNlCiAgICAgICAgbmV3X2xheWVyLmJpYXMuY29weV8oYi5jb250aWd1b3VzKCkpCiAgICAgICAgbmV3X2xheWVyLmJpYXMucmVxdWlyZXNfZ3JhZCA9IFRydWUKICAgIHJldHVybiBuZXdfbGF5ZXIKCmRlZiBfbGF2aXNfYXBwbHlfY2h1bmtpbmdfdG9fZm9yd2FyZChmb3J3YXJkX2ZuLCBjaHVua19zaXplLCBjaHVua19kaW0sICppbnB1dF90ZW5zb3JzKToKICAgIHJldHVybiBmb3J3YXJkX2ZuKCppbnB1dF90ZW5zb3JzKQoKZmluZF9wcnVuZWFibGVfaGVhZHNfYW5kX2luZGljZXMgPSBfbGF2aXNfZmluZF9wcnVuZWFibGVfaGVhZHNfYW5kX2luZGljZXMKcHJ1bmVfbGluZWFyX2xheWVyID0gX2xhdmlzX3BydW5lX2xpbmVhcl9sYXllcgphcHBseV9jaHVua2luZ190b19mb3J3YXJkID0gX2xhdmlzX2FwcGx5X2NodW5raW5nX3RvX2ZvcndhcmQKQUxMX0xBWUVSTk9STV9MQVlFUlMgPSBbbm4uTGF5ZXJOb3JtXQoKZGVmIF9hc3NlcnRfZGV2aWNlX21hcChkZXZpY2VfbWFwLCBudW1fYmxvY2tzKTogcGFzcwpkZWYgX2dldF9kZXZpY2VfbWFwKG5fbGF5ZXJzLCBkZXZpY2VzKTogcmV0dXJuIHtpOiBkZXZpY2VzWzBdIGZvciBpIGluIHJhbmdlKG5fbGF5ZXJzKX0KCmltcG9ydCB0cmFuc2Zvcm1lcnMKaW1wb3J0IHRyYW5zZm9ybWVycy5tb2RlbGluZ191dGlscwppZiBoYXNhdHRyKHRyYW5zZm9ybWVycy5tb2RlbGluZ191dGlscywgIlByZVRyYWluZWRNb2RlbCIpOgogICAgaWYgaGFzYXR0cih0cmFuc2Zvcm1lcnMubW9kZWxpbmdfdXRpbHMuUHJlVHJhaW5lZE1vZGVsLCAicmVzaXplX3Rva2VuX2VtYmVkZGluZ3MiKToKICAgICAgICBfb3JpZ19yID0gdHJhbnNmb3JtZXJzLm1vZGVsaW5nX3V0aWxzLlByZVRyYWluZWRNb2RlbC5yZXNpemVfdG9rZW5fZW1iZWRkaW5ncwogICAgICAgIGRlZiBfc2FmZV9yKHNlbGYsIG5ld19udW1fdG9rZW5zPU5vbmUsIHBhZF90b19tdWx0aXBsZV9vZj1Ob25lLCBtZWFuX3Jlc2l6aW5nPUZhbHNlLCAqKmt3YXJncyk6CiAgICAgICAgICAgIHRyeTogcmV0dXJuIF9vcmlnX3Ioc2VsZiwgbmV3X251bV90b2tlbnM9bmV3X251bV90b2tlbnMsIHBhZF90b19tdWx0aXBsZV9vZj1wYWRfdG9fbXVsdGlwbGVfb2YsIG1lYW5fcmVzaXppbmc9RmFsc2UsICoqa3dhcmdzKQogICAgICAgICAgICBleGNlcHQgVHlwZUVycm9yOiByZXR1cm4gX29yaWdfcihzZWxmLCBuZXdfbnVtX3Rva2Vucz1uZXdfbnVtX3Rva2VucywgcGFkX3RvX211bHRpcGxlX29mPXBhZF90b19tdWx0aXBsZV9vZiwgKiprd2FyZ3MpCiAgICAgICAgdHJhbnNmb3JtZXJzLm1vZGVsaW5nX3V0aWxzLlByZVRyYWluZWRNb2RlbC5yZXNpemVfdG9rZW5fZW1iZWRkaW5ncyA9IF9zYWZlX3IKICAgIGlmIGhhc2F0dHIodHJhbnNmb3JtZXJzLm1vZGVsaW5nX3V0aWxzLlByZVRyYWluZWRNb2RlbCwgIm1hcmtfdGllZF93ZWlnaHRzX2FzX2luaXRpYWxpemVkIik6CiAgICAgICAgX29yaWdfbSA9IHRyYW5zZm9ybWVycy5tb2RlbGluZ191dGlscy5QcmVUcmFpbmVkTW9kZWwubWFya190aWVkX3dlaWdodHNfYXNfaW5pdGlhbGl6ZWQKICAgICAgICBkZWYgX3NhZmVfbShzZWxmLCBpbmZvKToKICAgICAgICAgICAgdHJ5OiByZXR1cm4gX29yaWdfbShzZWxmLCBpbmZvKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOiByZXR1cm4gTm9uZQogICAgICAgIHRyYW5zZm9ybWVycy5tb2RlbGluZ191dGlscy5QcmVUcmFpbmVkTW9kZWwubWFya190aWVkX3dlaWdodHNfYXNfaW5pdGlhbGl6ZWQgPSBfc2FmZV9tCiAgICBpZiBoYXNhdHRyKHRyYW5zZm9ybWVycy5tb2RlbGluZ191dGlscy5QcmVUcmFpbmVkTW9kZWwsICJfZmluYWxpemVfbW9kZWxfbG9hZGluZyIpOgogICAgICAgIF9vcmlnX2YgPSB0cmFuc2Zvcm1lcnMubW9kZWxpbmdfdXRpbHMuUHJlVHJhaW5lZE1vZGVsLl9maW5hbGl6ZV9tb2RlbF9sb2FkaW5nCiAgICAgICAgZGVmIF9zYWZlX2YoY2xzX29yX3NlbGYsIG1vZGVsLCBsb2FkX2NvbmZpZz1Ob25lLCBsb2FkaW5nX2luZm89Tm9uZSwgKmFyZ3MsICoqa3dhcmdzKToKICAgICAgICAgICAgdHJ5OiByZXR1cm4gX29yaWdfZihtb2RlbCwgbG9hZF9jb25maWcsIGxvYWRpbmdfaW5mbywgKmFyZ3MsICoqa3dhcmdzKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOiByZXR1cm4gbG9hZGluZ19pbmZvCiAgICAgICAgdHJhbnNmb3JtZXJzLm1vZGVsaW5nX3V0aWxzLlByZVRyYWluZWRNb2RlbC5fZmluYWxpemVfbW9kZWxfbG9hZGluZyA9IF9zYWZlX2YKIyAtLS0tIGVuZCBzaGltIC0tLS0KJycnCgpkZWYgcmVhZChwKToKICAgIHJldHVybiBvcGVuKHAsICdyJywgZW5jb2Rpbmc9J3V0Zi04JywgZXJyb3JzPSdyZXBsYWNlJykucmVhZCgpCgpkZWYgd3JpdGUocCwgY29udGVudCk6CiAgICB3aXRoIG9wZW4ocCwgJ3cnLCBlbmNvZGluZz0ndXRmLTgnKSBhcyBmOgogICAgICAgIGYud3JpdGUoY29udGVudCkKICAgIHByaW50KGYiICBQYXRjaGVkOiB7cH0iKQoKZGVmIGluc2VydF9zaGltX2FmdGVyX2RvY3N0cmluZyhjb250ZW50LCBzaGltKToKICAgIGxpbmVzID0gY29udGVudC5zcGxpdCgnXG4nKQogICAgaSA9IDAKICAgIGlmIGxpbmVzIGFuZCBsaW5lc1swXS5zdGFydHN3aXRoKCcjIGNvZGluZycpOgogICAgICAgIGkgPSAxCiAgICB3aGlsZSBpIDwgbGVuKGxpbmVzKSBhbmQgKGxpbmVzW2ldLnN0cmlwKCkuc3RhcnRzd2l0aCgnIycpIG9yIGxpbmVzW2ldLnN0cmlwKCkgPT0gJycpOgogICAgICAgIGkgKz0gMQogICAgaWYgaSA8IGxlbihsaW5lcykgYW5kIGxpbmVzW2ldLnN0cmlwKCkuc3RhcnRzd2l0aCgnIiIiJyk6CiAgICAgICAgaSArPSAxCiAgICAgICAgd2hpbGUgaSA8IGxlbihsaW5lcykgYW5kICciIiInIG5vdCBpbiBsaW5lc1tpXToKICAgICAgICAgICAgaSArPSAxCiAgICAgICAgaSArPSAxCiAgICBsaW5lcy5pbnNlcnQoaSwgc2hpbSkKICAgIHJldHVybiAnXG4nLmpvaW4obGluZXMpCgojIDEuIG1vZGVsaW5nX3Q1LnB5CnQ1X3BhdGggPSBvcy5wYXRoLmpvaW4oTEFWSVNfRElSLCAibW9kZWxzL2JsaXAyX21vZGVscy9tb2RlbGluZ190NS5weSIpCmlmIG9zLnBhdGguZXhpc3RzKHQ1X3BhdGgpOgogICAgYyA9IHJlYWQodDVfcGF0aCkKICAgIGlmICcjIC0tLS0gdHJhbnNmb3JtZXJzIGNvbXBhdGliaWxpdHkgc2hpbScgbm90IGluIGM6CiAgICAgICAgYyA9IHJlLnN1YihyJ2Zyb20gdHJhbnNmb3JtZXJzXC51dGlsc1wubW9kZWxfcGFyYWxsZWxfdXRpbHMgaW1wb3J0W15cbl0rXG4nLCAnJywgYykKICAgICAgICBjID0gcmUuc3ViKHInZnJvbSB0cmFuc2Zvcm1lcnNcLnB5dG9yY2hfdXRpbHMgaW1wb3J0IFwoW14pXSpcKVxuJywgJycsIGMsIGZsYWdzPXJlLkRPVEFMTCkKICAgICAgICBjID0gcmUuc3ViKHInZnJvbSB0cmFuc2Zvcm1lcnNcLnB5dG9yY2hfdXRpbHMgaW1wb3J0IFteXG5dK1xuJywgJycsIGMpCiAgICAgICAgYyA9IHJlLnN1YihyJ2Zyb20gdHJhbnNmb3JtZXJzXC5tb2RlbGluZ191dGlscyBpbXBvcnQgXChbXildKlwpXG4nLCAnZnJvbSB0cmFuc2Zvcm1lcnMubW9kZWxpbmdfdXRpbHMgaW1wb3J0IFByZVRyYWluZWRNb2RlbFxuJywgYywgZmxhZ3M9cmUuRE9UQUxMKQogICAgICAgIGMgPSByZS5zdWIocicsP1xzKmlzX3RvcmNoX2Z4X3Byb3h5JywgJycsIGMpCiAgICAgICAgYyA9IHJlLnN1YihyJ1xzKkRVTU1ZX0lOUFVUUyxcbicsICdcbicsIGMpCiAgICAgICAgYyA9IHJlLnN1YihyJ1xzKkRVTU1ZX01BU0ssXG4nLCAnXG4nLCBjKQogICAgICAgIGMgPSBjLnJlcGxhY2UoJ2Fzc2VydF9kZXZpY2VfbWFwJywgJ19hc3NlcnRfZGV2aWNlX21hcCcpLnJlcGxhY2UoJ2dldF9kZXZpY2VfbWFwJywgJ19nZXRfZGV2aWNlX21hcCcpCiAgICAgICAgYyA9IGluc2VydF9zaGltX2FmdGVyX2RvY3N0cmluZyhjLCBTSElNKQogICAgICAgIHdyaXRlKHQ1X3BhdGgsIGMpCgojIDIuIG1lZC5weQptZWRfcGF0aCA9IG9zLnBhdGguam9pbihMQVZJU19ESVIsICJtb2RlbHMvbWVkLnB5IikKaWYgb3MucGF0aC5leGlzdHMobWVkX3BhdGgpOgogICAgYyA9IHJlYWQobWVkX3BhdGgpCiAgICBjID0gYy5yZXBsYWNlKCdmcm9tIHRyYW5zZm9ybWVycy5maWxlX3V0aWxzIGltcG9ydCcsICdmcm9tIHRyYW5zZm9ybWVycy51dGlscyBpbXBvcnQnKQogICAgYyA9IHJlLnN1YihyJ2Zyb20gdHJhbnNmb3JtZXJzXC5tb2RlbGluZ191dGlscyBpbXBvcnQgXChbXildKlwpXG4nLCAnZnJvbSB0cmFuc2Zvcm1lcnMubW9kZWxpbmdfdXRpbHMgaW1wb3J0IFByZVRyYWluZWRNb2RlbFxuJywgYywgZmxhZ3M9cmUuRE9UQUxMKQogICAgaWYgJyMgLS0tLSB0cmFuc2Zvcm1lcnMgY29tcGF0aWJpbGl0eSBzaGltJyBub3QgaW4gYzoKICAgICAgICBjID0gaW5zZXJ0X3NoaW1fYWZ0ZXJfZG9jc3RyaW5nKGMsIFNISU0pCiAgICB3cml0ZShtZWRfcGF0aCwgYykKCiMgMy4gUWZvcm1lci5weQpxZl9wYXRoID0gb3MucGF0aC5qb2luKExBVklTX0RJUiwgIm1vZGVscy9ibGlwMl9tb2RlbHMvUWZvcm1lci5weSIpCmlmIG9zLnBhdGguZXhpc3RzKHFmX3BhdGgpOgogICAgYyA9IHJlYWQocWZfcGF0aCkKICAgIGMgPSBjLnJlcGxhY2UoJ2Zyb20gdHJhbnNmb3JtZXJzLmZpbGVfdXRpbHMgaW1wb3J0JywgJ2Zyb20gdHJhbnNmb3JtZXJzLnV0aWxzIGltcG9ydCcpCiAgICBjID0gcmUuc3ViKHInZnJvbSB0cmFuc2Zvcm1lcnNcLm1vZGVsaW5nX3V0aWxzIGltcG9ydCBcKFteKV0qXClcbicsICdmcm9tIHRyYW5zZm9ybWVycy5tb2RlbGluZ191dGlscyBpbXBvcnQgUHJlVHJhaW5lZE1vZGVsXG4nLCBjLCBmbGFncz1yZS5ET1RBTEwpCiAgICBpZiAnIyAtLS0tIHRyYW5zZm9ybWVycyBjb21wYXRpYmlsaXR5IHNoaW0nIG5vdCBpbiBjOgogICAgICAgIGMgPSBpbnNlcnRfc2hpbV9hZnRlcl9kb2NzdHJpbmcoYywgU0hJTSkKICAgIHdyaXRlKHFmX3BhdGgsIGMpCgojIDQuIHFmb3JtZXJfSVQySVRfcmVyYW5rZXIucHkgKG1lYW5fcmVzaXppbmc9RmFsc2Ugb24gcmVzaXplX3Rva2VuX2VtYmVkZGluZ3MpCnJlcmFua2VyX3BhdGggPSBvcy5wYXRoLmpvaW4oTEFWSVNfRElSLCAibW9kZWxzL2JsaXAyX21vZGVscy9xZm9ybWVyX0lUMklUX3JlcmFua2VyLnB5IikKaWYgb3MucGF0aC5leGlzdHMocmVyYW5rZXJfcGF0aCk6CiAgICBjID0gcmVhZChyZXJhbmtlcl9wYXRoKQogICAgaWYgInJlc2l6ZV90b2tlbl9lbWJlZGRpbmdzKGxlbihzZWxmLnRva2VuaXplcikpIiBpbiBjOgogICAgICAgIGMgPSBjLnJlcGxhY2UoInJlc2l6ZV90b2tlbl9lbWJlZGRpbmdzKGxlbihzZWxmLnRva2VuaXplcikpIiwgInJlc2l6ZV90b2tlbl9lbWJlZGRpbmdzKGxlbihzZWxmLnRva2VuaXplciksIG1lYW5fcmVzaXppbmc9RmFsc2UpIikKICAgICAgICB3cml0ZShyZXJhbmtlcl9wYXRoLCBjKQoKIyA1LiBubHZyX2VuY29kZXIucHkKbmx2cl9wYXRoID0gb3MucGF0aC5qb2luKExBVklTX0RJUiwgIm1vZGVscy9ibGlwX21vZGVscy9ubHZyX2VuY29kZXIucHkiKQppZiBvcy5wYXRoLmV4aXN0cyhubHZyX3BhdGgpOgogICAgYyA9IHJlYWQobmx2cl9wYXRoKQogICAgYyA9IHJlLnN1YihyJ2Zyb20gdHJhbnNmb3JtZXJzXC5tb2RlbGluZ191dGlscyBpbXBvcnQgXChbXildKlwpXG4nLCAnZnJvbSB0cmFuc2Zvcm1lcnMubW9kZWxpbmdfdXRpbHMgaW1wb3J0IFByZVRyYWluZWRNb2RlbFxuJywgYywgZmxhZ3M9cmUuRE9UQUxMKQogICAgaWYgJyMgLS0tLSB0cmFuc2Zvcm1lcnMgY29tcGF0aWJpbGl0eSBzaGltJyBub3QgaW4gYzoKICAgICAgICBjID0gaW5zZXJ0X3NoaW1fYWZ0ZXJfZG9jc3RyaW5nKGMsIFNISU0pCiAgICB3cml0ZShubHZyX3BhdGgsIGMpCgojIDYuIG1vZGVscy9fX2luaXRfXy5weQppbml0X3BhdGggPSBvcy5wYXRoLmpvaW4oTEFWSVNfRElSLCAibW9kZWxzL19faW5pdF9fLnB5IikKaWYgb3MucGF0aC5leGlzdHMoaW5pdF9wYXRoKToKICAgIGMgPSByZWFkKGluaXRfcGF0aCkKICAgIGxpbmVzID0gYy5zcGxpdCgnXG4nKQogICAgbmV3X2xpbmVzID0gW10KICAgIGZvciBsaW5lIGluIGxpbmVzOgogICAgICAgIHMgPSBsaW5lLnN0cmlwKCkKICAgICAgICBpZiAocy5zdGFydHN3aXRoKCdmcm9tIGxhdmlzLm1vZGVscy4nKSBhbmQgCiAgICAgICAgICAgICdxZm9ybWVyJyBub3QgaW4gcy5sb3dlcigpIGFuZCAKICAgICAgICAgICAgJ2Jhc2VfbW9kZWwnIG5vdCBpbiBzLmxvd2VyKCkgYW5kIAogICAgICAgICAgICBub3Qgcy5zdGFydHN3aXRoKCd0cnk6JykpOgogICAgICAgICAgICBuZXdfbGluZXMuYXBwZW5kKGYidHJ5OlxuICAgIHtsaW5lfVxuZXhjZXB0IEV4Y2VwdGlvbjpcbiAgICBwYXNzIikKICAgICAgICBlbHNlOgogICAgICAgICAgICBuZXdfbGluZXMuYXBwZW5kKGxpbmUpCiAgICBjID0gJ1xuJy5qb2luKG5ld19saW5lcykKICAgIGZpbHRlcl9jb2RlID0gIlxuIyBGaWx0ZXIgX19hbGxfXyB0byBvbmx5IGV4aXN0aW5nIGF0dHJpYnV0ZXMgdG8gcHJldmVudCBBdHRyaWJ1dGVFcnJvciBvbiBzdGFyIGltcG9ydFxuX19hbGxfXyA9IFtrIGZvciBrIGluIF9fYWxsX18gaWYgayBpbiBnbG9iYWxzKCkgb3IgayBpbiBsb2NhbHMoKV1cbiIKICAgIGlmICJGaWx0ZXIgX19hbGxfXyB0byBvbmx5IGV4aXN0aW5nIGF0dHJpYnV0ZXMiIG5vdCBpbiBjOgogICAgICAgIGMgPSBjICsgZmlsdGVyX2NvZGUKICAgIHdyaXRlKGluaXRfcGF0aCwgYykKCiMgNy4gbGF2aXMvX19pbml0X18ucHkKcGtnX2luaXRfcGF0aCA9IG9zLnBhdGguam9pbihMQVZJU19ESVIsICJfX2luaXRfXy5weSIpCmlmIG9zLnBhdGguZXhpc3RzKHBrZ19pbml0X3BhdGgpOgogICAgYyA9IHJlYWQocGtnX2luaXRfcGF0aCkKICAgIHN0YXJfaW1wb3J0cyA9IFsKICAgICAgICAiZnJvbSBsYXZpcy5kYXRhc2V0cy5idWlsZGVycyBpbXBvcnQgKiIsCiAgICAgICAgImZyb20gbGF2aXMubW9kZWxzIGltcG9ydCAqIiwKICAgICAgICAiZnJvbSBsYXZpcy5wcm9jZXNzb3JzIGltcG9ydCAqIiwKICAgICAgICAiZnJvbSBsYXZpcy50YXNrcyBpbXBvcnQgKiIKICAgIF0KICAgIGZvciBpbXAgaW4gc3Rhcl9pbXBvcnRzOgogICAgICAgIGlmIGltcCBpbiBjIGFuZCBmInRyeTpcbiAgICB7aW1wfSIgbm90IGluIGM6CiAgICAgICAgICAgIGMgPSBjLnJlcGxhY2UoaW1wLCBmInRyeTpcbiAgICB7aW1wfVxuZXhjZXB0IEV4Y2VwdGlvbjpcbiAgICBwYXNzIikKICAgIHdyaXRlKHBrZ19pbml0X3BhdGgsIGMpCgpwcmludCgiXG4iICsgIj0iKjYwKQpwcmludCgiQWxsIExBVklTIHNvdXJjZSBmaWxlcyBwYXRjaGVkIHN1Y2Nlc3NmdWxseSEiKQpwcmludCgiPSIqNjApCg=="
patcher_path = "/kaggle/working/patch_lavis_compat.py"
with open(patcher_path, "wb") as f:
    f.write(base64.b64decode(PATCHER_B64))
lavis_dir = os.path.join(OMGM_REPO_DIR, "lavis")
!python {patcher_path} {lavis_dir}

OMGM Repository mounted from: /kaggle/working/OMGM
  Patched: /kaggle/working/OMGM/lavis/models/med.py
  Patched: /kaggle/working/OMGM/lavis/models/blip2_models/Qformer.py
  Patched: /kaggle/working/OMGM/lavis/models/blip_models/nlvr_encoder.py
  Patched: /kaggle/working/OMGM/lavis/models/__init__.py
  Patched: /kaggle/working/OMGM/lavis/__init__.py

All LAVIS source files patched successfully!


### Cell 4: Candidate Image Extraction & Title-to-Image Indexing (Requirement 2)
Extracts the candidate image folders into scratch space (`/kaggle/temp`) to prevent exceeding `/kaggle/working`'s 20GB disk limit, and constructs a fast title-to-image path lookup dictionary.

In [30]:
# ---------------------------------------------------------------------------
# Candidate Image Extraction & Title-to-Image Indexing
# ---------------------------------------------------------------------------
import zipfile

def prepare_candidate_images_dir(imgs_path, extract_dir="/kaggle/temp/wiki_imgs"):
    """Handles directory or zip archive for candidate images."""
    if os.path.isdir(imgs_path) and len(os.listdir(imgs_path)) > 0:
        return imgs_path
    if os.path.isfile(imgs_path) and imgs_path.endswith((".zip", ".tar", ".gz")):
        if os.path.isdir(extract_dir) and len(os.listdir(extract_dir)) > 100:
            print(f"Candidate images already extracted in {extract_dir}.")
            return extract_dir
        print(f"Extracting {imgs_path} -> {extract_dir}...")
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(imgs_path, 'r') as zf:
            zf.extractall(extract_dir)
        print(f"Extracted {len(os.listdir(extract_dir)):,} files.")
        return extract_dir
    return imgs_path

CANDIDATE_IMG_ROOT = prepare_candidate_images_dir(WIKI_IMGS_DIR)

print(f"Indexing available entity candidate images from {CANDIDATE_IMG_ROOT}...")
title2wikiimg = {}
valid_exts = (".jpg", ".jpeg", ".png", ".webp", ".JPG", ".JPEG", ".PNG")

# Step 1: Pre-index all image files in CANDIDATE_IMG_ROOT for O(1) lookup
img_file_lookup = {}
if os.path.isdir(CANDIDATE_IMG_ROOT):
    for root, dirs, files in os.walk(CANDIDATE_IMG_ROOT):
        for f in files:
            if f.endswith(valid_exts):
                full_p = os.path.join(root, f)
                # Map plain filename
                if f not in img_file_lookup:
                    img_file_lookup[f] = full_p
                # Map relative path from CANDIDATE_IMG_ROOT
                rel_p = os.path.relpath(full_p, CANDIDATE_IMG_ROOT).replace("\\", "/")
                img_file_lookup[rel_p] = full_p

print(f"Found {len(img_file_lookup):,} candidate image files on disk.")

# Step 2: Use candidate_wiki_images_max3.csv to map entity title -> image path
if os.path.isfile(CANDIDATE_IMAGES_CSV):
    print(f"Parsing candidate manifest from {CANDIDATE_IMAGES_CSV}...")
    import csv
    with open(CANDIDATE_IMAGES_CSV, "r", encoding="utf-8", errors="replace") as f:
        reader = csv.reader(f)
        header = next(reader, None)
        title_idx, path_idx = 0, 2
        if header:
            lower_h = [h.strip().lower() for h in header]
            if "title" in lower_h:
                title_idx = lower_h.index("title")
            for col in ["relative_path", "img_path", "filename", "path", "download_url"]:
                if col in lower_h:
                    path_idx = lower_h.index(col)
                    break
        f.seek(0)
        reader = csv.reader(f)
        if header:
            next(reader)
        for row in reader:
            if not row or len(row) <= title_idx:
                continue
            title = row[title_idx].strip()
            if not title or title in title2wikiimg:
                continue
            rel_path = row[path_idx].strip() if len(row) > path_idx else ""
            base_name = os.path.basename(rel_path) if rel_path else ""
            
            resolved_p = None
            if rel_path in img_file_lookup:
                resolved_p = img_file_lookup[rel_path]
            elif base_name in img_file_lookup:
                resolved_p = img_file_lookup[base_name]
            elif os.path.isfile(os.path.join(CANDIDATE_IMG_ROOT, rel_path)):
                resolved_p = os.path.join(CANDIDATE_IMG_ROOT, rel_path)
            elif os.path.isfile(os.path.join(CANDIDATE_IMG_ROOT, base_name)):
                resolved_p = os.path.join(CANDIDATE_IMG_ROOT, base_name)
                
            if resolved_p and os.path.exists(resolved_p):
                title2wikiimg[title] = resolved_p

# Step 3: Directory walk for folders named by entity title (e.g. root/<title>/*.jpg)
if os.path.isdir(CANDIDATE_IMG_ROOT):
    for root, dirs, files in os.walk(CANDIDATE_IMG_ROOT):
        imgs = [f for f in files if f.endswith(valid_exts)]
        if imgs:
            folder_title = os.path.basename(root).strip()
            if folder_title and folder_title not in title2wikiimg:
                title2wikiimg[folder_title] = os.path.join(root, sorted(imgs)[0])
            # Also check normalized title (replace '_' with ' ')
            space_title = folder_title.replace("_", " ")
            if space_title not in title2wikiimg:
                title2wikiimg[space_title] = os.path.join(root, sorted(imgs)[0])

print(f"Indexed {len(title2wikiimg):,} entity titles with valid images.")


Indexing available entity candidate images from /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/wiki_imgs/wiki_imgs/full/wiki_image_split...
Found 39,878 candidate image files on disk.
Parsing candidate manifest from /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/candidate_wiki_images_max3.csv...
Indexed 11,143 entity titles with valid images.


### Cell 5: Load Knowledge Base & Stage 1 Results (Requirement 1 & 4)

In [31]:
class WikipediaKnowledgeBaseEntry:
    """Represents a single Wikipedia article entry in the knowledge base."""
    def __init__(self, entry_dict):
        self.title = entry_dict.get("title", "")
        self.url = entry_dict.get("url", "")
        self.image_urls = entry_dict.get("image_urls", [])
        self.image_reference_descriptions = entry_dict.get("image_reference_descriptions", [])
        self.image_section_indices = entry_dict.get("image_section_indices", [])
        self.section_titles = entry_dict.get("section_titles", [])
        self.section_texts = entry_dict.get("section_texts", [])
        self.image = {}
        self.score = {}

def reconstruct_wiki_sections(knowledge_entry):
    """Reconstruct sections from knowledge base entry, matching OMGM's section formatting."""
    title = knowledge_entry.title
    sections = []
    for it, section_title in enumerate(knowledge_entry.section_titles):
        sec_lower = section_title.lower()
        if "external links" in sec_lower or "references" in sec_lower:
            continue
        section_text = knowledge_entry.section_texts[it] if it < len(knowledge_entry.section_texts) else ""
        sec_str = f"# Wiki Article: {title}\n## Section Title: {section_title}\n{section_text}"
        sections.append(sec_str)
    return sections

print(f"Loading Knowledge Base from {KB_PATH}...")
with open(KB_PATH, "r", encoding="utf-8") as f:
    kb_dict = json.load(f)
knowledge_base_dict = {entry["url"]: WikipediaKnowledgeBaseEntry(entry) for entry in kb_dict.values()}
print(f"Loaded Knowledge Base with {len(knowledge_base_dict):,} entries.")
del kb_dict
gc.collect()

print(f"Loading Stage 1 results from {STAGE1_RESULT_PATH}...")
with open(STAGE1_RESULT_PATH, "r", encoding="utf-8") as f:
    stage1_results = json.load(f)
print(f"Loaded Stage 1 outputs for {len(stage1_results):,} queries.")

# Build Query ID -> Dataset Image ID mapping from sample test CSV
qid_to_img_id = {}
if SAMPLE_FILE and os.path.exists(SAMPLE_FILE):
    import csv
    with open(SAMPLE_FILE, "r", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        for row in reader:
            did = row.get("data_id")
            img_id = row.get("dataset_image_ids") or row.get("image_id") or row.get("dataset_image_id")
            if did and img_id:
                qid_to_img_id[str(did).strip()] = str(img_id).split("|")[0].strip()
print(f"Loaded {len(qid_to_img_id):,} query-to-image mappings from sample CSV.")

# Coverage Verification: Check how many candidate entities have images
unique_candidate_titles = set()
for q_id, q_data in stage1_results.items():
    for ent in q_data["retrieved_entities"]:
        unique_candidate_titles.add(ent["title"].strip())

matched_titles = [t for t in unique_candidate_titles if t in title2wikiimg]
coverage_pct = (len(matched_titles) / max(len(unique_candidate_titles), 1)) * 100
print(f"Candidate Image Coverage: {len(matched_titles):,} / {len(unique_candidate_titles):,} ({coverage_pct:.2f}%)")


Loading Knowledge Base from /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/infoseek_100k_wiki/wiki_100_dict_v4.json...
Loaded Knowledge Base with 100,000 entries.
Loading Stage 1 results from /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/result.json...
Loaded Stage 1 outputs for 1,000 queries.
Loaded 1,000 query-to-image mappings from sample CSV.
Candidate Image Coverage: 10,625 / 11,657 (91.15%)


### Cell 6: Model & Preprocessor Initialization (Requirement 3)
Initializes `TargetPad(1.25, 224)` image transform, loads `QformerIT2ITReranker`, and restores weights from checkpoint.

In [32]:
import sys
import os
import re
import warnings
import importlib
import PIL
from PIL import Image
import torch
import torch.nn as nn
from torchvision.transforms import Compose, Resize, CenterCrop, ToTensor, Normalize
import torchvision.transforms.functional as F

warnings.filterwarnings("ignore")

# ===========================================================================
# 1. Self-Healing Disk Cleanup: Fix models/__init__.py and strip bad shims
# ===========================================================================
omgm_lavis_dir = "/kaggle/working/OMGM/lavis"
if not os.path.exists(omgm_lavis_dir):
    omgm_lavis_dir = os.path.abspath("OMGM Main Baseline/lavis")

clean_init_code = '''"""
 Copyright (c) 2022, salesforce.com, inc.
 All rights reserved.
 SPDX-License-Identifier: BSD-3-Clause
 For full license text, see the LICENSE file in the repo root or https://opensource.org/licenses/BSD-3-Clause
"""

import logging
import torch
from omegaconf import OmegaConf
from lavis.common.registry import registry
from lavis.models.base_model import BaseModel
from lavis.processors.base_processor import BaseProcessor

# Safe model imports
model_imports = [
    ("lavis.models.albef_models.albef_classification", "AlbefClassification"),
    ("lavis.models.albef_models.albef_feature_extractor", "AlbefFeatureExtractor"),
    ("lavis.models.albef_models.albef_nlvr", "AlbefNLVR"),
    ("lavis.models.albef_models.albef_pretrain", "AlbefPretrain"),
    ("lavis.models.albef_models.albef_retrieval", "AlbefRetrieval"),
    ("lavis.models.albef_models.albef_vqa", "AlbefVQA"),
    ("lavis.models.alpro_models.alpro_qa", "AlproQA"),
    ("lavis.models.alpro_models.alpro_retrieval", "AlproRetrieval"),
    ("lavis.models.blip2_models.blip2", "Blip2Base"),
    ("lavis.models.blip2_models.blip2_image_text_matching", "Blip2ITM"),
    ("lavis.models.blip2_models.blip2_opt", "Blip2OPT"),
    ("lavis.models.blip2_models.blip2_qformer", "Blip2Qformer"),
    ("lavis.models.blip2_models.blip2_t5", "Blip2T5"),
    ("lavis.models.blip2_models.blip2_t5_instruct", "Blip2T5Instruct"),
    ("lavis.models.blip2_models.blip2_vicuna_instruct", "Blip2VicunaInstruct"),
    ("lavis.models.blip2_models.qformer_IT2IT_reranker", "QformerIT2ITReranker"),
    ("lavis.models.blip2_models.qformer_T2IT_reranker", "QformerT2ITReranker"),
    ("lavis.models.blip2_models.qformer_T2T_reranker", "QformerT2TReranker"),
    ("lavis.models.blip_models.blip", "BlipBase"),
    ("lavis.models.blip_models.blip_caption", "BlipCaption"),
    ("lavis.models.blip_models.blip_cir_base", "BlipCirBase"),
    ("lavis.models.blip_models.blip_classification", "BlipClassification"),
    ("lavis.models.blip_models.blip_feature_extractor", "BlipFeatureExtractor"),
    ("lavis.models.blip_models.blip_image_text_matching", "BlipITM"),
    ("lavis.models.blip_models.blip_nlvr", "BlipNLVR"),
    ("lavis.models.blip_models.blip_pretrain", "BlipPretrain"),
    ("lavis.models.blip_models.blip_retrieval", "BlipRetrieval"),
    ("lavis.models.blip_models.blip_vqa", "BlipVQA"),
    ("lavis.models.clip_models.model", "CLIP"),
    ("lavis.models.gpt_models.gpt_dialogue", "GPTDialogue"),
    ("lavis.models.img2prompt_models.img2prompt_vqa", "Img2PromptVQA"),
    ("lavis.models.med", "XBertLMHeadDecoder"),
    ("lavis.models.pnp_vqa_models.pnp_unifiedqav2_fid", "PNPUnifiedQAv2FiD"),
    ("lavis.models.pnp_vqa_models.pnp_vqa", "PNPVQA"),
    ("lavis.models.vit", "VisionTransformerEncoder"),
]

for mod_name, cls_name in model_imports:
    try:
        mod = __import__(mod_name, fromlist=[cls_name])
        globals()[cls_name] = getattr(mod, cls_name)
    except Exception:
        pass

__all__ = [
    "load_model",
    "load_preprocess",
    "load_model_and_preprocess",
    "BaseModel",
    "AlbefClassification",
    "AlbefFeatureExtractor",
    "AlbefNLVR",
    "AlbefPretrain",
    "AlbefRetrieval",
    "AlbefVQA",
    "AlproQA",
    "AlproRetrieval",
    "BlipBase",
    "BlipCaption",
    "BlipCirBase",
    "BlipClassification",
    "BlipFeatureExtractor",
    "BlipITM",
    "BlipNLVR",
    "BlipPretrain",
    "BlipRetrieval",
    "BlipVQA",
    "QformerIT2ITReranker",
    "QformerT2TReranker",
    "QformerT2ITReranker",
    "Blip2Base",
    "Blip2ITM",
    "Blip2OPT",
    "Blip2T5",
    "Blip2T5Instruct",
    "Blip2VicunaInstruct",
    "PNPVQA",
    "Img2PromptVQA",
    "PNPUnifiedQAv2FiD",
    "CLIP",
    "VisionTransformerEncoder",
    "XBertLMHeadDecoder",
    "GPTDialogue",
]

def load_model(name, model_type, is_eval=False, device="cpu", checkpoint=None):
    model = registry.get_model_class(name).from_pretrained(model_type=model_type)
    if checkpoint is not None:
        model.load_checkpoint(checkpoint)
    if is_eval:
        model.eval()
    if device == "cpu":
        model = model.float()
    return model.to(device)

def load_preprocess(config):
    def _build_proc_from_cfg(cfg):
        return (
            registry.get_processor_class(cfg.name).from_config(cfg)
            if cfg is not None
            else BaseProcessor()
        )
    vis_processors = dict()
    txt_processors = dict()
    vis_proc_cfg = config.get("vis_processor")
    txt_proc_cfg = config.get("text_processor")
    if vis_proc_cfg is not None:
        vis_train_cfg = vis_proc_cfg.get("train")
        vis_eval_cfg = vis_proc_cfg.get("eval")
    else:
        vis_train_cfg = None
        vis_eval_cfg = None
    vis_processors["train"] = _build_proc_from_cfg(vis_train_cfg)
    vis_processors["eval"] = _build_proc_from_cfg(vis_eval_cfg)
    if txt_proc_cfg is not None:
        txt_train_cfg = txt_proc_cfg.get("train")
        txt_eval_cfg = txt_proc_cfg.get("eval")
    else:
        txt_train_cfg = None
        txt_eval_cfg = None
    txt_processors["train"] = _build_proc_from_cfg(txt_train_cfg)
    txt_processors["eval"] = _build_proc_from_cfg(txt_eval_cfg)
    return vis_processors, txt_processors

def load_model_and_preprocess(name, model_type, is_eval=False, device="cpu"):
    model_cls = registry.get_model_class(name)
    model = model_cls.from_pretrained(model_type=model_type)
    if is_eval:
        model.eval()
    cfg = OmegaConf.load(model_cls.default_config_path(model_type))
    if cfg is not None:
        preprocess_cfg = cfg.preprocess
        vis_processors, txt_processors = load_preprocess(preprocess_cfg)
    else:
        vis_processors, txt_processors = None, None
    if device == "cpu" or device == torch.device("cpu"):
        model = model.float()
    return model.to(device), vis_processors, txt_processors

class ModelZoo:
    def __init__(self) -> None:
        self.model_zoo = {
            k: list(v.PRETRAINED_MODEL_CONFIG_DICT.keys())
            for k, v in registry.mapping.get("model_name_mapping", {}).items()
        }
    def __str__(self) -> str:
        return "=" * 50 + "\\n" + f"{'Architectures':<30} {'Types'}\\n" + "=" * 50
    def __iter__(self):
        return iter(self.model_zoo.items())
    def __len__(self):
        return sum([len(v) for v in self.model_zoo.values()])

model_zoo = ModelZoo()
__all__ = [k for k in __all__ if k in globals() or k in locals()]
'''

if os.path.exists(omgm_lavis_dir):
    models_init_file = os.path.join(omgm_lavis_dir, "models/__init__.py")
    try:
        with open(models_init_file, "w", encoding="utf-8") as fp:
            fp.write(clean_init_code)
    except Exception as e:
        print(f"Notice: models/__init__.py write: {e}")

    for root, dirs, files in os.walk(omgm_lavis_dir):
        for f in files:
            if f.endswith(".py"):
                fpath = os.path.join(root, f)
                try:
                    with open(fpath, "r", encoding="utf-8", errors="replace") as fp:
                        content = fp.read()
                    changed = False
                    if "_safe_from_pretrained" in content:
                        content = re.sub(r'if hasattr\(transformers\.modeling_utils\.PreTrainedModel, "from_pretrained"\):.*?_is_safe_from_pretrained_patched = True\n', '', content, flags=re.DOTALL)
                        changed = True
                    if "_safe_r" in content:
                        content = re.sub(r'_orig_r = transformers\.modeling_utils\.PreTrainedModel\.resize_token_embeddings.*?transformers\.modeling_utils\.PreTrainedModel\._is_safe_resize_patched = True\n', '', content, flags=re.DOTALL)
                        changed = True
                    if changed:
                        with open(fpath, "w", encoding="utf-8") as fp:
                            fp.write(content)
                except Exception:
                    pass

# ===========================================================================
# 2. Reset Kernel Module Cache & Restore Pristine HuggingFace Classes
# ===========================================================================
import transformers
import transformers.modeling_utils
importlib.reload(transformers.modeling_utils)
importlib.reload(transformers)

transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = {}

if hasattr(transformers.modeling_utils.PreTrainedModel, "tie_weights"):
    _orig_tie = transformers.modeling_utils.PreTrainedModel.tie_weights
    def _safe_tie(self, *args, **kwargs):
        if not hasattr(self, "all_tied_weights_keys"):
            self.all_tied_weights_keys = {}
        try:
            return _orig_tie(self, *args, **kwargs)
        except Exception:
            return None
    transformers.modeling_utils.PreTrainedModel.tie_weights = _safe_tie

for k in list(sys.modules.keys()):
    if "lavis" in k:
        del sys.modules[k]

# ===========================================================================
# 3. Dynamic Bridges for PyTorch 2.x & Modern Transformers 4.45+
# ===========================================================================

def _safe_get_input_embeddings(self):
    if hasattr(self, "bert") and hasattr(self.bert, "embeddings") and hasattr(self.bert.embeddings, "word_embeddings"):
        return self.bert.embeddings.word_embeddings
    if hasattr(self, "embeddings") and hasattr(self.embeddings, "word_embeddings"):
        return self.embeddings.word_embeddings
    if hasattr(self, "word_embeddings"):
        return self.word_embeddings
    for child in self.children() if hasattr(self, "children") else []:
        if hasattr(child, "word_embeddings"):
            return child.word_embeddings
        if hasattr(child, "embeddings") and hasattr(child.embeddings, "word_embeddings"):
            return child.embeddings.word_embeddings
    base = getattr(self, getattr(self, "base_model_prefix", ""), None)
    if base is not None and base is not self and hasattr(base, "get_input_embeddings"):
        return base.get_input_embeddings()
    for name, m in self.named_modules() if hasattr(self, "named_modules") else []:
        if isinstance(m, nn.Embedding) and ("word" in name or "embed" in name):
            return m
    raise NotImplementedError(f"Cannot find input embeddings for {self.__class__.__name__}")

def _safe_set_input_embeddings(self, value):
    if hasattr(self, "bert") and hasattr(self.bert, "embeddings") and hasattr(self.bert.embeddings, "word_embeddings"):
        self.bert.embeddings.word_embeddings = value
        return
    if hasattr(self, "embeddings") and hasattr(self.embeddings, "word_embeddings"):
        self.embeddings.word_embeddings = value
        return
    if hasattr(self, "word_embeddings"):
        self.word_embeddings = value
        return
    for child in self.children() if hasattr(self, "children") else []:
        if hasattr(child, "word_embeddings"):
            child.word_embeddings = value
            return
        if hasattr(child, "embeddings") and hasattr(child.embeddings, "word_embeddings"):
            child.embeddings.word_embeddings = value
            return
    base = getattr(self, getattr(self, "base_model_prefix", ""), None)
    if base is not None and base is not self and hasattr(base, "set_input_embeddings"):
        base.set_input_embeddings(value)
        return
    raise NotImplementedError(f"Cannot set input embeddings for {self.__class__.__name__}")

transformers.modeling_utils.PreTrainedModel.get_input_embeddings = _safe_get_input_embeddings
transformers.modeling_utils.PreTrainedModel.set_input_embeddings = _safe_set_input_embeddings

if not getattr(torch.nn.Module, "_is_safe_load_patched", False):
    _orig_load_state_dict = torch.nn.Module.load_state_dict
    def _safe_load_state_dict(self, state_dict, *args, **kwargs):
        strict = kwargs.get("strict", True)
        if len(args) > 0:
            strict = args[0]
        if not strict and isinstance(state_dict, dict):
            model_state = self.state_dict()
            clean_dict = {}
            for k, v in state_dict.items():
                if k in model_state:
                    if hasattr(v, "shape") and hasattr(model_state[k], "shape"):
                        if v.shape == model_state[k].shape:
                            clean_dict[k] = v
                        else:
                            print(f"[SafeLoad] Skipping '{k}': shape mismatch (checkpoint {v.shape} vs model {model_state[k].shape})")
                    else:
                        clean_dict[k] = v
                else:
                    clean_dict[k] = v
            state_dict = clean_dict
        return _orig_load_state_dict(self, state_dict, *args, **kwargs)
    torch.nn.Module.load_state_dict = _safe_load_state_dict
    torch.nn.Module._is_safe_load_patched = True

if not getattr(torch.nn.Module, "_is_safe_to_patched", False):
    _orig_to = torch.nn.Module.to
    def _safe_to(self, *args, **kwargs):
        def _materialize(mod):
            for name, buf in list(mod._buffers.items()):
                if buf is not None and getattr(buf, "is_meta", False):
                    mod._buffers[name] = torch.zeros(buf.shape, dtype=buf.dtype, device="cpu")
            for name, param in list(mod._parameters.items()):
                if param is not None and getattr(param, "is_meta", False):
                    mod._parameters[name] = nn.Parameter(torch.zeros(param.shape, dtype=param.dtype, device="cpu"))
            for child in mod.children():
                _materialize(child)
        _materialize(self)
        return _orig_to(self, *args, **kwargs)
    torch.nn.Module.to = _safe_to
    torch.nn.Module._is_safe_to_patched = True

if not getattr(transformers.modeling_utils.PreTrainedModel, "_is_safe_resize_patched", False):
    _orig_resize = transformers.modeling_utils.PreTrainedModel.resize_token_embeddings
    def _safe_resize(self, new_num_tokens=None, pad_to_multiple_of=None, mean_resizing=False, **kwargs):
        try:
            res = _orig_resize(self, new_num_tokens=new_num_tokens, pad_to_multiple_of=pad_to_multiple_of, mean_resizing=False, **kwargs)
        except TypeError:
            res = _orig_resize(self, new_num_tokens=new_num_tokens, pad_to_multiple_of=pad_to_multiple_of, **kwargs)
        if new_num_tokens is not None and hasattr(self, "cls") and hasattr(self.cls, "predictions"):
            preds = self.cls.predictions
            if hasattr(preds, "bias") and preds.bias is not None and preds.bias.shape[0] != new_num_tokens:
                new_b = nn.Parameter(torch.zeros(new_num_tokens, device=preds.bias.device, dtype=preds.bias.dtype))
                n = min(preds.bias.shape[0], new_num_tokens)
                new_b.data[:n] = preds.bias.data[:n]
                preds.bias = new_b
                if hasattr(preds, "decoder"):
                    preds.decoder.bias = preds.bias
        return res
    transformers.modeling_utils.PreTrainedModel.resize_token_embeddings = _safe_resize
    transformers.modeling_utils.PreTrainedModel._is_safe_resize_patched = True

import lavis
from lavis.common.registry import registry
if registry.get_path("library_root") is None:
    lavis_root = os.path.dirname(os.path.abspath(lavis.__file__))
    registry.register_path("library_root", lavis_root)
    registry.register_path("repo_root", os.path.abspath(os.path.join(lavis_root, "..")))
    registry.register_path("cache_root", os.path.abspath(os.path.join(lavis_root, "../cache")))

from lavis.models.blip2_models.Qformer import BertLMHeadModel, BertModel, BertPreTrainedModel, BertConfig

BertLMHeadModel.all_tied_weights_keys = {}
BertModel.all_tied_weights_keys = {}
BertPreTrainedModel.all_tied_weights_keys = {}

BertLMHeadModel.get_input_embeddings = lambda self: self.bert.embeddings.word_embeddings
BertLMHeadModel.set_input_embeddings = lambda self, val: setattr(self.bert.embeddings, "word_embeddings", val)
BertModel.get_input_embeddings = lambda self: self.embeddings.word_embeddings
BertModel.set_input_embeddings = lambda self, val: setattr(self.embeddings, "word_embeddings", val)

@classmethod
def _blm_from_pretrained(cls, pretrained_model_name_or_path, *model_args, **kwargs):
    config = kwargs.get("config", None)
    if config is None and len(model_args) > 0:
        config = model_args[0]
    model = cls(config)
    try:
        from transformers import BertModel as HFBertModel
        hf_bert = HFBertModel.from_pretrained(pretrained_model_name_or_path)
        model.bert.load_state_dict(hf_bert.state_dict(), strict=False)
    except Exception:
        pass
    return model

BertLMHeadModel.from_pretrained = _blm_from_pretrained

# ===========================================================================
# 4. Model Loading & Inference Setup
# ===========================================================================
from lavis.models import load_model_and_preprocess

class TargetPad:
    """Aspect-ratio preserving padding matching OMGM's data_utils."""
    def __init__(self, target_ratio: float, size: int):
        self.size = size
        self.target_ratio = target_ratio

    def __call__(self, image):
        w, h = image.size
        actual_ratio = max(w, h) / max(min(w, h), 1)
        if actual_ratio < self.target_ratio:
            return image
        scaled_max_wh = max(w, h) / self.target_ratio
        hp = max(int((scaled_max_wh - w) / 2), 0)
        vp = max(int((scaled_max_wh - h) / 2), 0)
        return F.pad(image, [hp, vp, hp, vp], 0, "constant")

def targetpad_transform(target_ratio: float, dim: int):
    return Compose([
        TargetPad(target_ratio, dim),
        Resize(dim, interpolation=PIL.Image.BICUBIC),
        CenterCrop(dim),
        lambda img: img.convert("RGB"),
        ToTensor(),
        Normalize((0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)),
    ])

preprocess = targetpad_transform(1.25, 224)

# Robust fallback check for RERANKER_CKPT_PATH
if 'RERANKER_CKPT_PATH' not in globals() or not RERANKER_CKPT_PATH or not os.path.isfile(str(RERANKER_CKPT_PATH)):
    for cand in [
        "/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/reranker_model.pth/reranker_model.pth/model_23912.pth",
        "/kaggle/input/agentic-vqa-ds/reranker_model.pth/reranker_model.pth/model_23912.pth",
    ]:
        if os.path.isfile(cand):
            RERANKER_CKPT_PATH = cand
            break
    if not RERANKER_CKPT_PATH or not os.path.isfile(str(RERANKER_CKPT_PATH)):
        import glob
        matches = glob.glob("/kaggle/input/**/model_23912.pth", recursive=True)
        if matches:
            RERANKER_CKPT_PATH = matches[0]

print("Instantiating QformerIT2ITReranker...")
reranker_model, vis_processors, txt_processors = load_model_and_preprocess(
    name="qformer_IT2IT_reranker",
    model_type="pretrain",
    is_eval=True,
    device="cpu",
)

if txt_processors is None or "eval" not in txt_processors:
    txt_processors = {"eval": lambda x: str(x).lower().strip()}

if RERANKER_CKPT_PATH and os.path.exists(RERANKER_CKPT_PATH):
    print(f"Loading checkpoint weights from: {RERANKER_CKPT_PATH}")
    checkpoint = torch.load(RERANKER_CKPT_PATH, map_location="cpu")
    if "model" in checkpoint:
        checkpoint = checkpoint["model"]
    msg = reranker_model.load_state_dict(checkpoint, strict=False)
    print(f"Loaded weights (missing keys: {len(msg.missing_keys)}, unexpected keys: {len(msg.unexpected_keys)})")
else:
    print("Warning: No checkpoint loaded! Inference will run with initialized weights.")

reranker_model = reranker_model.half().to(device).eval()
print(f"Reranker model ready in FP16 on {device}.")

Instantiating QformerIT2ITReranker...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading checkpoint weights from: /kaggle/input/datasets/alphacentauriv/agentic-vqa-ds/reranker_model.pth/reranker_model.pth/model_23912.pth
Loaded weights (missing keys: 0, unexpected keys: 0)
Reranker model ready in FP16 on cuda:0.


### Cell 7: Helper Functions & Recall Evaluation Metric

In [37]:
def eval_recall(candidates, ground_truth, top_ks=[1, 5, 10, 20]):
    recall = {k: 0 for k in top_ks}
    for k in top_ks:
        if ground_truth in candidates[:k]:
            recall[k] = 1
    return recall

def get_query_image_path(query_key, search_dir, qid_map=None):
    candidates = []
    if qid_map and str(query_key) in qid_map:
        candidates.append(qid_map[str(query_key)])
    clean_k = str(query_key).split("|")[0].strip()
    candidates.append(clean_k)
    
    for raw_id in candidates:
        clean_id = raw_id.strip()
        for ext in ["", ".jpg", ".JPEG", ".png", ".JPG", ".PNG"]:
            p = os.path.join(search_dir, f"{clean_id}{ext}")
            if os.path.isfile(p):
                return p
                
    # Fallback partial match search in search_dir
    for raw_id in candidates:
        matches = glob.glob(os.path.join(search_dir, f"*{raw_id}*"))
        for m in matches:
            if os.path.isfile(m) and m.lower().endswith((".jpg", ".jpeg", ".png", ".JPEG", ".PNG")):
                return m
                
    raise FileNotFoundError(f"Query image for query '{query_key}' not found in {search_dir}")


### Cell 8: Pilot Test on 10–20 Queries (Requirement 6)
Runs inference on a small slice of 20 queries to validate runtime latency, GPU VRAM footprint, and output formatting.

In [39]:
# ---------------------------------------------------------------------------
# Compatibility Bridge: ensure get_head_mask is present on active model
# ---------------------------------------------------------------------------
def _get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
    if head_mask is not None:
        if head_mask.dim() == 1:
            head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
            head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
        elif head_mask.dim() == 2:
            head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
        if is_attention_chunked:
            head_mask = head_mask.unsqueeze(-1)
        head_mask = head_mask.to(dtype=self.dtype)
    else:
        head_mask = [None] * num_hidden_layers
    return head_mask

try:
    from lavis.models.blip2_models.Qformer import BertModel, BertLMHeadModel
    BertModel.get_head_mask = _get_head_mask
    BertLMHeadModel.get_head_mask = _get_head_mask
except Exception:
    pass

if hasattr(reranker_model, "Qformer") and hasattr(reranker_model.Qformer, "bert"):
    reranker_model.Qformer.bert.get_head_mask = _get_head_mask.__get__(reranker_model.Qformer.bert)

# ---------------------------------------------------------------------------
# Pilot Stage 2 Reranking Execution (20 Queries)
# ---------------------------------------------------------------------------
import time

PILOT_SAMPLE_COUNT = 20
STEP1_ALPHA = 0.9
RERANK_CHUNK_SIZE = 128  # Memory-safe chunk size for 16GB T4

pilot_query_ids = list(stage1_results.keys())[:PILOT_SAMPLE_COUNT]
print(f"Running Pilot Stage 2 Reranking on {len(pilot_query_ids)} queries...")

pilot_recalls_step1 = {k: 0 for k in [1, 5, 10, 20]}
pilot_recalls_step2 = {k: 0 for k in [1, 5, 10, 20]}

torch.cuda.reset_peak_memory_stats(device)
start_time = time.time()

for it, q_id in enumerate(pilot_query_ids):
    q_data = stage1_results[q_id]
    ground_truth_url = q_data["ground_truth_url"]
    question_text = txt_processors["eval"](q_data["question"])
    step1_entities = q_data["retrieved_entities"]
    step1_sims = q_data["retrieval_similarities"]
    
    # Load and preprocess query image
    try:
        q_img_path = get_query_image_path(q_id, QUERY_IMG_DIR, qid_to_img_id)
        query_img_pil = PIL.Image.open(q_img_path).convert("RGB")
        ref_img_tensor = preprocess(query_img_pil).to(device).half().unsqueeze(0)
    except Exception as img_err:
        print(f"Warning: Failed to load query image for {q_id}: {img_err}. Using blank query image tensor.")
        ref_img_tensor = torch.zeros((1, 3, 224, 224), device=device, dtype=torch.float16)
    
    # Collect candidate images and sections
    IT2IT_null = []
    candidate_images = []
    candidate_sections = []
    entry2ITindex = {}
    
    for entry_idx, ent in enumerate(step1_entities):
        title = ent["title"].strip()
        url = ent["url"].strip()
        
        if title not in title2wikiimg or not os.path.exists(title2wikiimg[title]):
            IT2IT_null.append(True)
            continue
            
        if url not in knowledge_base_dict:
            IT2IT_null.append(True)
            continue
            
        kb_entry = knowledge_base_dict[url]
        entry_sections = reconstruct_wiki_sections(kb_entry)
        if len(entry_sections) == 0:
            IT2IT_null.append(True)
            continue
            
        img_path = title2wikiimg[title]
        try:
            cand_img_pil = PIL.Image.open(img_path).convert("RGB")
            candidate_images.extend([cand_img_pil] * len(entry_sections))
            candidate_sections.extend([txt_processors["eval"](s) for s in entry_sections])
            start_idx = len(candidate_sections) - len(entry_sections)
            entry2ITindex[entry_idx] = [start_idx + i for i in range(len(entry_sections))]
            IT2IT_null.append(False)
        except Exception as cand_err:
            IT2IT_null.append(True)
        
    # Inference & late-interaction
    with torch.no_grad(), torch.cuda.amp.autocast():
        query_fusion_embs = reranker_model.extract_features(
            {"image": ref_img_tensor, "text_input": question_text},
            mode="multimodal_query",
        )["multimodal_embeds"]
        
        if len(candidate_images) > 0:
            cand_fusion_list = []
            for chunk_start in range(0, len(candidate_images), RERANK_CHUNK_SIZE):
                img_chunk = candidate_images[chunk_start : chunk_start + RERANK_CHUNK_SIZE]
                sec_chunk = candidate_sections[chunk_start : chunk_start + RERANK_CHUNK_SIZE]
                
                proc_chunk = torch.stack([preprocess(img) for img in img_chunk]).to(device).half()
                embs_chunk = reranker_model.extract_features(
                    {"image": proc_chunk, "text_input": sec_chunk},
                    mode="multimodal_candidate",
                )["multimodal_embeds"]
                cand_fusion_list.append(embs_chunk)
                
            all_cand_embs = torch.cat(cand_fusion_list, dim=0)
            
            # Late-interaction MaxSim scoring: S = (1/32) * sum_i max_j (Q_i . C_j)
            sim_matrix = torch.matmul(query_fusion_embs, all_cand_embs.permute(0, 2, 1))
            chunk_scores = sim_matrix.max(dim=2).values.sum(dim=1).squeeze(0)
            
            # Aggregate section scores per candidate entity
            final_scores = []
            for entry_idx in range(len(step1_entities)):
                if entry_idx in entry2ITindex and len(entry2ITindex[entry_idx]) > 0:
                    indices = entry2ITindex[entry_idx]
                    mm_score = float(chunk_scores[indices].max().cpu().item())
                else:
                    mm_score = -999.0
                
                s1_score = float(step1_sims[entry_idx])
                hybrid_score = (STEP1_ALPHA * s1_score) + ((1.0 - STEP1_ALPHA) * mm_score)
                final_scores.append(hybrid_score)
        else:
            final_scores = [float(s) for s in step1_sims]
            
    # Rank candidates by hybrid score
    sorted_indices = sorted(range(len(final_scores)), key=lambda k: final_scores[k], reverse=True)
    reranked_urls = [step1_entities[i]["url"] for i in sorted_indices]
    step1_urls = [ent["url"] for ent in step1_entities]
    
    # Compute Pilot Recalls
    for k in [1, 5, 10, 20]:
        if ground_truth_url in step1_urls[:k]:
            pilot_recalls_step1[k] += 1
        if ground_truth_url in reranked_urls[:k]:
            pilot_recalls_step2[k] += 1

elapsed = time.time() - start_time
peak_mem_gb = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
latency_per_query = elapsed / len(pilot_query_ids)

print("\n" + "="*60)
print("PILOT EVALUATION SUMMARY (20 Queries)")
print("="*60)
print(f"Elapsed Time:      {elapsed:.2f}s ({latency_per_query:.2f}s per query)")
print(f"Peak GPU VRAM:     {peak_mem_gb:.2f} GB")
print("-" * 60)
for k in [1, 5, 10, 20]:
    r1 = pilot_recalls_step1[k] / len(pilot_query_ids) * 100
    r2 = pilot_recalls_step2[k] / len(pilot_query_ids) * 100
    delta = r2 - r1
    print(f"Recall@{k:<2}: Stage 1 = {r1:5.1f}%  -->  Stage 2 = {r2:5.1f}%  (Δ = {delta:+5.1f}%)")
print("="*60)

Running Pilot Stage 2 Reranking on 20 queries...

PILOT EVALUATION SUMMARY (20 Queries)
Elapsed Time:      356.59s (17.83s per query)
Peak GPU VRAM:     7.00 GB
------------------------------------------------------------
Recall@1 : Stage 1 =  35.0%  -->  Stage 2 =  30.0%  (Δ =  -5.0%)
Recall@5 : Stage 1 =  50.0%  -->  Stage 2 =  65.0%  (Δ = +15.0%)
Recall@10: Stage 1 =  70.0%  -->  Stage 2 =  70.0%  (Δ =  +0.0%)
Recall@20: Stage 1 =  75.0%  -->  Stage 2 =  75.0%  (Δ =  +0.0%)


### Cell 9: Full 1,000-Query Reranking with Resumable Checkpoints (Requirement 5 & 6)
Processes all 1,000 queries with intermediate checkpointing every 50 iterations, so runs can resume seamlessly without lost compute.

In [40]:
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "result_checkpoint.json")
FINAL_RESULT_FILE = os.path.join(OUTPUT_DIR, "result.json")
SAVE_INTERVAL = 50

# Load existing checkpoint if resuming
stage2_results = {}
if os.path.exists(FINAL_RESULT_FILE):
    print(f"Found completed results file at {FINAL_RESULT_FILE}.")
    with open(FINAL_RESULT_FILE, "r", encoding="utf-8") as f:
        stage2_results = json.load(f)
    print(f"Loaded {len(stage2_results)} processed queries.")
elif os.path.exists(CHECKPOINT_FILE):
    print(f"Found intermediate checkpoint at {CHECKPOINT_FILE}.")
    with open(CHECKPOINT_FILE, "r", encoding="utf-8") as f:
        stage2_results = json.load(f)
    print(f"Resuming from query {len(stage2_results)}...")

all_query_ids = list(stage1_results.keys())
recalls_step2 = {k: 0 for k in [1, 5, 10, 20]}

print(f"\nExecuting Stage 2 Reranking on all {len(all_query_ids)} queries...")
for it, q_id in tqdm(enumerate(all_query_ids), total=len(all_query_ids), desc="Stage 2 Reranking"):
    if q_id in stage2_results:
        # Already computed
        q_out = stage2_results[q_id]
        cand_urls = [e["url"] for e in q_out["retrieved_entities"]]
        rec = eval_recall(cand_urls, stage1_results[q_id]["ground_truth_url"])
        for k in [1, 5, 10, 20]:
            recalls_step2[k] += rec[k]
        continue
        
    q_data = stage1_results[q_id]
    ground_truth_url = q_data["ground_truth_url"]
    question_text = txt_processors["eval"](q_data["question"])
    step1_entities = q_data["retrieved_entities"]
    step1_sims = q_data["retrieval_similarities"]
    
    # Load query image
    try:
        q_img_path = get_query_image_path(q_id, QUERY_IMG_DIR, qid_to_img_id)
        query_img_pil = PIL.Image.open(q_img_path).convert("RGB")
        ref_img_tensor = preprocess(query_img_pil).to(device).half().unsqueeze(0)
    except Exception as img_err:
        print(f"Warning: Failed to load query image for {q_id}: {img_err}. Using blank query image tensor.")
        ref_img_tensor = torch.zeros((1, 3, 224, 224), device=device, dtype=torch.float16)
    
    # Collect candidate images and sections
    IT2IT_null = []
    candidate_images = []
    candidate_sections = []
    entry2ITindex = {}
    
    for entry_idx, ent in enumerate(step1_entities):
        title = ent["title"].strip()
        url = ent["url"].strip()
        
        if title not in title2wikiimg or not os.path.exists(title2wikiimg[title]):
            IT2IT_null.append(True)
            continue
            
        if url not in knowledge_base_dict:
            IT2IT_null.append(True)
            continue
            
        kb_entry = knowledge_base_dict[url]
        entry_sections = reconstruct_wiki_sections(kb_entry)
        if len(entry_sections) == 0:
            IT2IT_null.append(True)
            continue
            
        img_path = title2wikiimg[title]
        try:
            cand_img_pil = PIL.Image.open(img_path).convert("RGB")
            candidate_images.extend([cand_img_pil] * len(entry_sections))
            candidate_sections.extend([txt_processors["eval"](s) for s in entry_sections])
            start_idx = len(candidate_sections) - len(entry_sections)
            entry2ITindex[entry_idx] = [start_idx + i for i in range(len(entry_sections))]
            IT2IT_null.append(False)
        except Exception as cand_err:
            IT2IT_null.append(True)
        
    with torch.no_grad(), torch.cuda.amp.autocast():
        query_fusion_embs = reranker_model.extract_features(
            {"image": ref_img_tensor, "text_input": question_text},
            mode="multimodal_query",
        )["multimodal_embeds"]
        
        if len(candidate_images) > 0:
            cand_fusion_list = []
            for chunk_start in range(0, len(candidate_images), RERANK_CHUNK_SIZE):
                img_chunk = candidate_images[chunk_start : chunk_start + RERANK_CHUNK_SIZE]
                sec_chunk = candidate_sections[chunk_start : chunk_start + RERANK_CHUNK_SIZE]
                
                proc_chunk = torch.stack([preprocess(img) for img in img_chunk]).to(device).half()
                embs_chunk = reranker_model.extract_features(
                    {"image": proc_chunk, "text_input": sec_chunk},
                    mode="multimodal_candidate",
                )["multimodal_embeds"]
                cand_fusion_list.append(embs_chunk)
            
            cand_fusion_embs = torch.cat(cand_fusion_list, dim=0)
            sim_token2token = torch.matmul(query_fusion_embs, cand_fusion_embs.permute(0, 2, 1)).squeeze(0)
            sim_querytoken, _ = sim_token2token.max(-1)
            sim_q2c = sim_querytoken.sum(-1) / sim_querytoken.size(-1)
        else:
            sim_q2c = torch.tensor([], device=device)
            
        step2_sim = [0.0] * len(step1_sims)
        for entry_idx, is_null in enumerate(IT2IT_null):
            if is_null:
                continue
            step2_sim[entry_idx] = max([sim_q2c[i].item() for i in entry2ITindex[entry_idx]])
            
        alpha = STEP1_ALPHA
        combined_scores = (
            alpha * torch.tensor(step1_sims, device=device)
            + (1.0 - alpha) * torch.tensor(step2_sim, device=device)
        )
        sorted_scores, reranked_indices = torch.sort(combined_scores, descending=True)
        
    # Collect fine-grained section similarities per entity (required for Stage 3)
    top_sec_sim = []
    for i in reranked_indices:
        orig_idx = i.item()
        if orig_idx not in entry2ITindex:
            top_sec_sim.append([])
        else:
            top_sec_sim.append([sim_q2c[idx].item() for idx in entry2ITindex[orig_idx]])
            
    reranked_entities = [step1_entities[i] for i in reranked_indices]
    reranked_sims = [s.item() for s in sorted_scores]
    
    stage2_results[q_id] = {
        "question": q_data.get("question", ""),
        "ground_truth_url": ground_truth_url,
        "top1_entity_url": reranked_entities[0]["url"],
        "top1_entity_title": reranked_entities[0]["title"],
        "retrieved_entities": reranked_entities,
        "retrieval_similarities": reranked_sims,
        "sec_sim": top_sec_sim,
    }
    
    cand_urls = [e["url"] for e in reranked_entities]
    rec = eval_recall(cand_urls, ground_truth_url)
    for k in [1, 5, 10, 20]:
        recalls_step2[k] += rec[k]
        
    # Periodic intermediate checkpointing
    if (it + 1) % SAVE_INTERVAL == 0:
        with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
            json.dump(stage2_results, f, indent=2)

# Save final result.json
with open(FINAL_RESULT_FILE, "w", encoding="utf-8") as f:
    json.dump(stage2_results, f, indent=2)
print(f"\nSaved final Stage 2 results to: {FINAL_RESULT_FILE}")

total_queries = len(all_query_ids)
print("\n" + "="*45)
print(f"STAGE 2 RETRIEVAL EVALUATION RESULTS (N={total_queries})")
print("="*45)
for k in [1, 5, 10, 20]:
    avg_rec = (recalls_step2[k] / total_queries) * 100.0
    print(f"Recall@{k:2d}: {avg_rec:6.2f}% ({recalls_step2[k]}/{total_queries})")
print("="*45)



Executing Stage 2 Reranking on all 1000 queries...


Stage 2 Reranking: 100%|██████████| 1000/1000 [5:31:15<00:00, 19.88s/it] 



Saved final Stage 2 results to: /kaggle/working/step2_results/result.json

STAGE 2 RETRIEVAL EVALUATION RESULTS (N=1000)
Recall@ 1:  59.10% (591/1000)
Recall@ 5:  76.70% (767/1000)
Recall@10:  79.30% (793/1000)
Recall@20:  83.40% (834/1000)


### Cell 10: Deliverables Verification for Stage 3 Consumption
Verifies that the generated `result.json` satisfies all contract requirements for Stage 3 (presence of `top1_entity_url`, `sec_sim`, and aligned candidate counts).

In [41]:
with open(FINAL_RESULT_FILE, "r", encoding="utf-8") as f:
    verify_data = json.load(f)

sample_qid = list(verify_data.keys())[0]
sample_entry = verify_data[sample_qid]

print("Verification check on first query:")
print(f"Query ID:           {sample_qid}")
print(f"Top-1 Entity:       {sample_entry['top1_entity_title']} ({sample_entry['top1_entity_url']})")
print(f"Top-1 Similarity:   {sample_entry['retrieval_similarities'][0]:.4f}")
print(f"Candidate Entities: {len(sample_entry['retrieved_entities'])}")
print(f"Section Sim Lists:  {len(sample_entry['sec_sim'])}")
if sample_entry['sec_sim'] and len(sample_entry['sec_sim'][0]) > 0:
    print(f"Top-1 Section Count: {len(sample_entry['sec_sim'][0])} (first sec sim: {sample_entry['sec_sim'][0][0]:.4f})")

assert len(verify_data) == len(stage1_results), "Mismatch in query count!"
assert "top1_entity_url" in sample_entry, "Missing top1_entity_url!"
assert "sec_sim" in sample_entry, "Missing sec_sim!"
print("\nAll Stage 3 delivery requirements validated successfully!")


Verification check on first query:
Query ID:           infoseek_val_00000053
Top-1 Entity:       Canadian National Vimy Memorial (https://en.wikipedia.org/wiki/Canadian_National_Vimy_Memorial)
Top-1 Similarity:   0.4354
Candidate Entities: 20
Section Sim Lists:  20
Top-1 Section Count: 23 (first sec sim: 1.9022)

All Stage 3 delivery requirements validated successfully!
